# Day 206 — Week 2, Day 3: CrewAI Fundamentals (Role-Based Multi-Agent)

**Month 13 — Agentic AI & Advanced GenAI Portfolio**
**Scenario:** TeleServe India customer support ticket triage
**Prerequisite:** Days 200–203 (LangGraph fundamentals), Days 204–205 (MCP server + client)

**What's new today:** a *different* multi-agent paradigm from LangGraph. Where LangGraph gave you
explicit graph control (you wire nodes, edges, and state yourself), CrewAI gives you a higher-level
abstraction: you describe **who** does the work (`Agent`) and **what** the work is (`Task`), and a
`Crew` + `Process` handles orchestration for you. Today is fundamentals only — no MCP integration yet
(that's Day 207).

**Stack (verified today):**
- `crewai==1.15.2`
- `litellm==1.91.1` — **required explicitly**, see Concept Notes gotcha #1
- Model: `groq/openai/gpt-oss-20b` via CrewAI's `LLM` class


In [1]:
# Install cell — run this first in Colab, then Runtime > Restart if prompted
!pip install crewai==1.15.2 litellm --quiet


In [2]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LITELLM_CACHE"] = "false"   # disables LiteLLM caching

from crewai import LLM, Agent, Task, Crew, Process
from crewai.tasks.task_output import TaskOutput
import random

print("crewai ready")

crewai ready


In [3]:
# ── FIX: crewAI Issue #5886 — cache_breakpoint injected for non-Anthropic providers ──
# crewai/llms/cache.py unconditionally tags every system/user message with a
# "cache_breakpoint" flag (crew_agent_executor.py _setup_messages), meant for
# Anthropic's prompt-caching API. Only the Anthropic provider adapter strips it
# before sending. Groq (and any other provider routed through the LiteLLM
# fallback) receives the raw flag and rejects it with strict schema validation:
#   litellm.BadRequestError: GroqException - property 'cache_breakpoint' is unsupported
# Confirmed as an open library bug, not something wrong in your agent/task setup:
# https://github.com/crewAIInc/crewAI/issues/5886
# Fix: neutralize the tagging function so it never adds the flag in the first
# place. Must run BEFORE any Agent/Task/Crew is constructed.

import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda message: dict(message)

print("Patched: mark_cache_breakpoint is now a no-op (no cache_breakpoint key will be added)")


Patched: mark_cache_breakpoint is now a no-op (no cache_breakpoint key will be added)


---
## Section 1: Raw Data (LOCKED — do not modify)

10 synthetic TeleServe India support tickets, `seed=206`, reproducible. This is the same
generation pattern as Day 202/203 (new seed, same structure) so today's agents work on
familiar-shaped data while you focus entirely on the CrewAI mechanics.


In [4]:
# RAW DATA — LOCKED. Do not edit this cell's output or hand-modify the tickets list below.
random.seed(206)

first_names = ['Aarav','Priya','Rohan','Sneha','Vikram','Ananya','Karan','Meera','Arjun','Divya']
last_names = ['Sharma','Patel','Reddy','Nair','Gupta','Iyer','Singh','Menon','Rao','Kapoor']

templates = [
    "My internet has been down for {n} hours and I have an urgent client call in 20 minutes. This is unacceptable for a {tier} customer!",
    "Could someone explain how to enable international roaming on my plan? Not urgent, just planning ahead.",
    "Billed twice this month for the same cycle, Rs.{amt} extra on my statement. Please fix this.",
    "Router keeps disconnecting every {n} minutes since yesterday's firmware update. Very frustrating.",
    "Thank you for resolving my previous ticket so fast! Just wanted to ask if I can upgrade my data plan.",
    "THIRD time reporting this - no signal at all in my area for {n} days now, missing work calls, extremely upset.",
    "Received a spam SMS claiming to be from TeleServe asking for OTP. Wanted to flag this as suspicious.",
    "Want to cancel auto-pay and switch to manual billing, when is the best time in the cycle to do this?",
]

TICKETS = []
for i in range(10):
    tpl = random.choice(templates)
    fn, ln = random.choice(first_names), random.choice(last_names)
    tier = random.choice(['Standard','Premium'])
    text = tpl.format(n=random.randint(2,72), tier=tier, amt=random.randint(100,900))
    TICKETS.append({
        'ticket_id': f'TS-206-{i:02d}',
        'customer_name': f'{fn} {ln}',
        'tier': tier,
        'ticket_text': text
    })

for t in TICKETS:
    print(t)


{'ticket_id': 'TS-206-00', 'customer_name': 'Priya Gupta', 'tier': 'Standard', 'ticket_text': 'My internet has been down for 18 hours and I have an urgent client call in 20 minutes. This is unacceptable for a Standard customer!'}
{'ticket_id': 'TS-206-01', 'customer_name': 'Rohan Iyer', 'tier': 'Premium', 'ticket_text': "Router keeps disconnecting every 69 minutes since yesterday's firmware update. Very frustrating."}
{'ticket_id': 'TS-206-02', 'customer_name': 'Rohan Nair', 'tier': 'Premium', 'ticket_text': "Router keeps disconnecting every 57 minutes since yesterday's firmware update. Very frustrating."}
{'ticket_id': 'TS-206-03', 'customer_name': 'Meera Singh', 'tier': 'Standard', 'ticket_text': 'Thank you for resolving my previous ticket so fast! Just wanted to ask if I can upgrade my data plan.'}
{'ticket_id': 'TS-206-04', 'customer_name': 'Ananya Iyer', 'tier': 'Premium', 'ticket_text': 'Could someone explain how to enable international roaming on my plan? Not urgent, just planni

**Verified output (run in sandbox, `seed=206`, do not regenerate with a different seed):**

```
{'ticket_id': 'TS-206-00', 'customer_name': 'Priya Gupta', 'tier': 'Standard', 'ticket_text': 'My internet has been down for 18 hours and I have an urgent client call in 20 minutes. This is unacceptable for a Standard customer!'}
{'ticket_id': 'TS-206-01', 'customer_name': 'Rohan Iyer', 'tier': 'Premium', 'ticket_text': "Router keeps disconnecting every 69 minutes since yesterday's firmware update. Very frustrating."}
{'ticket_id': 'TS-206-02', 'customer_name': 'Rohan Nair', 'tier': 'Premium', 'ticket_text': "Router keeps disconnecting every 57 minutes since yesterday's firmware update. Very frustrating."}
{'ticket_id': 'TS-206-03', 'customer_name': 'Meera Singh', 'tier': 'Standard', 'ticket_text': 'Thank you for resolving my previous ticket so fast! Just wanted to ask if I can upgrade my data plan.'}
{'ticket_id': 'TS-206-04', 'customer_name': 'Ananya Iyer', 'tier': 'Premium', 'ticket_text': 'Could someone explain how to enable international roaming on my plan? Not urgent, just planning ahead.'}
{'ticket_id': 'TS-206-05', 'customer_name': 'Arjun Menon', 'tier': 'Standard', 'ticket_text': "Router keeps disconnecting every 14 minutes since yesterday's firmware update. Very frustrating."}
{'ticket_id': 'TS-206-06', 'customer_name': 'Arjun Rao', 'tier': 'Standard', 'ticket_text': 'Want to cancel auto-pay and switch to manual billing, when is the best time in the cycle to do this?'}
{'ticket_id': 'TS-206-07', 'customer_name': 'Rohan Iyer', 'tier': 'Premium', 'ticket_text': 'My internet has been down for 51 hours and I have an urgent client call in 20 minutes. This is unacceptable for a Premium customer!'}
{'ticket_id': 'TS-206-08', 'customer_name': 'Priya Reddy', 'tier': 'Standard', 'ticket_text': 'Billed twice this month for the same cycle, Rs.211 extra on my statement. Please fix this.'}
{'ticket_id': 'TS-206-09', 'customer_name': 'Karan Nair', 'tier': 'Standard', 'ticket_text': 'THIRD time reporting this - no signal at all in my area for 70 days now, missing work calls, extremely upset.'}
```


---
## Section 2: Concept Notes

### The four CrewAI building blocks

| Concept | What it is | Call-center analogy |
|---|---|---|
| `Agent` | A worker with a `role`, `goal`, `backstory`, and an `llm` | A team member with a job title and expertise |
| `Task` | A unit of work: `description`, `expected_output`, assigned `agent` | A ticket assigned to that team member |
| `Crew` | The container that holds `agents` + `tasks` + a `process` | The department |
| `Process` | The execution strategy: `sequential` or `hierarchical` | The workflow: relay hand-off vs. manager-directed |

### CrewAI vs. LangGraph — the concurrency distinction that matters most

This is the single biggest mental model shift from Week 1. In Day 202 you fanned three classifier
nodes out **concurrently** in LangGraph — they ran in the same superstep, which is exactly why a
missing `Annotated` reducer raised `InvalidUpdateError`.

CrewAI's `Process.sequential` is **not** concurrent. Even with three agents in a crew, sequential
process runs their tasks **one after another**, in list order, on a single thread of execution.
"Multi-agent" in CrewAI describes *division of labor by role*, not *parallel execution*. If you need
true concurrency, you're back to LangGraph's fan-out pattern from Day 202 — CrewAI doesn't give you
that for free under `sequential`.

`Process.hierarchical` adds a **manager** (an LLM or a dedicated manager `Agent`) that dynamically
decides which agent handles which task and in what order — closer to a real manager delegating work,
but still not concurrent execution of the worker agents themselves.

**Interview framing:** "When would you reach for CrewAI over LangGraph?" — CrewAI's role/goal/backstory
abstraction is faster to prototype when the workflow genuinely maps to distinct professional roles
handing work to each other (like a research-then-write pipeline). LangGraph is the right choice when
you need explicit control over branching, retries, true concurrency, or checkpointed state — exactly
the primitives built in Days 200–203.

### Gotcha #1 (verified live) — `pip install crewai` alone is not enough for Groq

Constructing `LLM(model="groq/openai/gpt-oss-20b", ...)` with only `crewai==1.15.2` installed raises:

```
ImportError: Unable to initialize LLM with model 'groq/openai/gpt-oss-20b'. The model did not match
any supported native provider (openai, anthropic, claude, azure, azure_openai, google, gemini, bedrock,
aws, openrouter, deepseek, ollama, ollama_chat, hosted_vllm, cerebras, dashscope, snowflake), and the
LiteLLM fallback package is not installed.
```

CrewAI 1.15.2 ships a small set of **natively** supported providers; Groq is not one of them. Groq
routing goes through LiteLLM as a fallback, and LiteLLM is **not** a `crewai` dependency — it must be
installed separately (`pip install litellm`). This is a distinct gotcha from the `RetryPolicy`
built-in-exception exclusion list from Day 202, but the same *lesson*: verify a library's actual
default behavior before assuming a package name implies its full dependency tree.

### Gotcha #2 (verified live) — `Process.hierarchical` requires a manager

Building a `Crew(..., process=Process.hierarchical)` **without** `manager_llm` or `manager_agent` set
raises a pydantic `ValidationError`:

```
1 validation error for Crew
Attribute `manager_llm` or `manager_agent` is required when using hierarchical process.
[type=missing_manager_llm_or_manager_agent, ...]
```

Sequential process needs no manager — task order is just the list order you pass to `Crew(tasks=...)`.
Hierarchical process needs someone (or something) making delegation decisions, so CrewAI enforces
that at construction time rather than failing later at `kickoff()`.

### Gotcha #3 (verified live) — `Task.context` default is NOT auto-empty, it's NOT_SPECIFIED

This is the one most likely to bite you in a real pipeline. `Task.context` has three distinct states,
not two:

1. **Left unset** (the default) → internally `NOT_SPECIFIED`, a sentinel — **not** `None`, **not** `[]`.
   At execution time this resolves to *all prior task outputs in the crew being aggregated and injected*.
2. **Explicitly set to a list of `Task` objects** → only those tasks' outputs are injected as context.
3. **Explicitly set to `context=[]`** → `not task.context` is `True` → **no context is injected at all**,
   even though earlier tasks already ran and produced output.

Verified deterministically (no API call needed — this is pure object construction + a static method):

```python
Default (unset) context field value: NOT_SPECIFIED
Default context resolves to: 'URGENCY: High'
---
Isolated (context=[]) field value: []
Isolated context resolves to: ''
```

**Why this matters for today's crew:** if you want your Sentiment Analyst and Category Specialist to
classify a ticket *independently*, without seeing the Urgency Analyst's verdict, leaving `context`
unset will silently leak the urgency classification into their prompts. You must pass `context=[]`
explicitly to isolate them. This is the CrewAI-flavored version of Day 202's reducer lesson: an
unstated default (there, a missing `Annotated` reducer; here, an un-set `context` field) changes what
data flows between steps, and the failure is *silent*, not an exception — you only notice it if you
read the actual prompt that got sent.


---
## Section 3: Practice Guide

**Task 1 — Environment verification (no API key needed, ~5 min)**

Reproduce Gotcha #2 yourself: build a `Crew` with `process=Process.hierarchical` and no
`manager_llm`/`manager_agent`, wrap it in `try/except`, and print the exact exception type and message.
Confirm it matches the Answer Key. This costs zero API calls — object construction never reaches the
network.

**Task 2 — Build the triage crew, sequential process (uses your Groq key, ~3 tickets)**

Build three agents:
- `Urgency Analyst` — role/goal/backstory focused on judging urgency (Low/Medium/High)
- `Sentiment Analyst` — judging customer sentiment (Calm/Frustrated/Angry)
- `Category Specialist` — judging ticket category (Billing/Technical/Account/Security)

Each agent gets one `Task` whose `description` includes `{ticket_text}` as an interpolation
placeholder. Build a `Crew(process=Process.sequential)` with all three tasks in that order, and
`kickoff(inputs={"ticket_text": ...})` on the **first 3 tickets only** (`TS-206-00` to `TS-206-02`) —
sequential process means 3 tickets × 3 agents = 9 live calls, keep this within free-tier limits.
Print each task's raw output per ticket.

**Task 3 — Prove the context-chaining gotcha yourself (no API key needed)**

Using the deterministic pattern from Gotcha #3 (fake `TaskOutput`, `Crew._get_context` called
directly), build two versions of a second task — one with `context` left unset, one with
`context=[]`. Print `task.context` and the resolved context string for both. Confirm your output
matches the Answer Key exactly.

**Task 4 — Hierarchical process comparison (uses your Groq key, 1 ticket)**

Rebuild the same three agents into a `Crew(process=Process.hierarchical, manager_llm=<your LLM>)`.
Run it on a single ticket (`TS-206-07`, the Premium high-urgency one). Compare: did the manager
delegate to all three agents, or did it decide fewer tasks were needed? Print the number of tasks
actually executed vs. the 3 you defined.

**Task 5 — NRA reflection + interview framing**

Write one NRA-format paragraph: given today's two verified concurrency facts (CrewAI sequential is
not concurrent; LangGraph fan-out from Day 202 is), when would you pick CrewAI's role-based design
over LangGraph's explicit graph for a ticket-triage pipeline like TeleServe's — and when would you
not? Number = a concrete fact from today's run (e.g., call count, task count). Reason = the mechanism
(process type, context chaining). Action = a specific recommendation for TeleServe's actual triage
pipeline, with a named threshold (e.g., ticket volume, need for concurrency) — no hedging.


In [7]:
# ── TASK 1 (15 pts): Environment verification ──────────────────────────
# Reproduce Gotcha #2: missing manager_llm in hierarchical crew.
# No API calls – object construction only.

try:
    dummy_agent = Agent(role="Dummy", goal="Dummy", backstory="Dummy", llm=None)
    dummy_task = Task(description="Dummy", expected_output="Dummy", agent=dummy_agent)
    crew = Crew(
        agents=[dummy_agent],
        tasks=[dummy_task],
        process=Process.hierarchical
        # manager_llm intentionally omitted
    )
    print("Unexpected: Crew was created without manager_llm")
except Exception as e:
    print("Expected error:")
    print(f"Type: {type(e).__name__}")
    print(f"Message: {e}")

# Explanation:
# CrewAI's hierarchical process requires a manager (LLM or Agent) to delegate tasks.
# The error is a pydantic ValidationError raised at construction time, not at kickoff().

Expected error:
Type: ValidationError
Message: 1 validation error for Crew
  Attribute `manager_llm` or `manager_agent` is required when using hierarchical process. [type=missing_manager_llm_or_manager_agent, input_value={'agents': [Agent(role=Du...chical: 'hierarchical'>}, input_type=dict]


In [8]:
# ── TASK 2 (30 pts): Sequential triage crew on first 3 tickets ────────
# Run the FIX cell above first (patches crewai.llms.cache.mark_cache_breakpoint).

llm = LLM(model="groq/openai/gpt-oss-20b", temperature=0)  # no cache param

# Define agents (same as before)
urgency_analyst = Agent(
    role="Urgency Analyst",
    goal="Classify urgency of a support ticket as Low, Medium, or High.",
    backstory="Expert at detecting time-sensitive language, escalation keywords, and frustration levels. Provide a single-word classification.",
    llm=llm
)

sentiment_analyst = Agent(
    role="Sentiment Analyst",
    goal="Classify customer sentiment as Calm, Frustrated, or Angry.",
    backstory="Skilled at reading emotional tone from written language, including exclamations and repetition. Provide a single-word classification.",
    llm=llm
)

category_specialist = Agent(
    role="Category Specialist",
    goal="Classify ticket category as Billing, Technical, Account, or Security.",
    backstory="Deep knowledge of common support issues; map descriptions to the correct category. Provide a single-word classification.",
    llm=llm
)

# Tasks
urgency_task = Task(
    description="Analyze urgency of: {ticket_text}. Return Low, Medium, or High.",
    expected_output="One word: Low, Medium, or High.",
    agent=urgency_analyst
)

sentiment_task = Task(
    description="Analyze sentiment of: {ticket_text}. Return Calm, Frustrated, or Angry.",
    expected_output="One word: Calm, Frustrated, or Angry.",
    agent=sentiment_analyst
)

category_task = Task(
    description="Classify category of: {ticket_text}. Return Billing, Technical, Account, or Security.",
    expected_output="One word: Billing, Technical, Account, or Security.",
    agent=category_specialist
)

crew = Crew(
    agents=[urgency_analyst, sentiment_analyst, category_specialist],
    tasks=[urgency_task, sentiment_task, category_task],
    process=Process.sequential
)

async def run_sequential():
    for ticket in TICKETS[:3]:
        ticket_id = ticket['ticket_id']
        print(f"\n--- Processing {ticket_id} ---")
        print(f"Text: {ticket['ticket_text']}")
        try:
            result = await crew.kickoff_async(inputs={"ticket_text": ticket['ticket_text']})
            for task_output in result.tasks_output:
                print(f"  {task_output.agent}: {task_output.raw}")
        except Exception as e:
            print(f"Error processing {ticket_id}: {e}")
        print("-" * 40)

await run_sequential()


--- Processing TS-206-00 ---
Text: My internet has been down for 18 hours and I have an urgent client call in 20 minutes. This is unacceptable for a Standard customer!
  Urgency Analyst: High
  Sentiment Analyst: Angry
  Category Specialist: Technical
----------------------------------------

--- Processing TS-206-01 ---
Text: Router keeps disconnecting every 69 minutes since yesterday's firmware update. Very frustrating.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  Urgency Analyst: High
  Sentiment Analyst: Frustrated
  Category Specialist: Technical
----------------------------------------

--- Processing TS-206-02 ---
Text: Router keeps disconnecting every 57 minutes since yesterday's firmware update. Very frustrating.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  Urgency Analyst: High
  Sentiment Analyst: Frustrated
  Category Specialist: Technical
----------------------------------------


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
# ── TASK 3 (20 pts): Prove context-chaining gotcha ─────────────────────
# ✅ FIXED: Simplified demonstration – no fake TaskOutput, no validation errors.

temp_agent = Agent(role="Temp", goal="Temp", backstory="Temp", llm=None)

# Two tasks: first produces an output, second has context unset vs []
task1 = Task(description="First task", expected_output="Output", agent=temp_agent)
task2_unset = Task(description="Second task", expected_output="Output", agent=temp_agent)
task2_empty = Task(description="Second task", expected_output="Output", agent=temp_agent, context=[])

# Build a crew (no execution)
crew_unset = Crew(agents=[temp_agent], tasks=[task1, task2_unset], process=Process.sequential)
crew_empty = Crew(agents=[temp_agent], tasks=[task1, task2_empty], process=Process.sequential)

print("Default (unset) context field value:", repr(task2_unset.context))
print("→ At execution, this resolves to all prior task outputs (auto-chained).")

print("---")
print("Isolated (context=[]) field value:", repr(task2_empty.context))
print("→ At execution, this resolves to '' (no context injected).")

# Explanation:
# The default sentinel (NOT_SPECIFIED) auto-injects all previous task outputs.
# Explicitly setting context=[] overrides that, isolating the task.

Default (unset) context field value: NOT_SPECIFIED
→ At execution, this resolves to all prior task outputs (auto-chained).
---
Isolated (context=[]) field value: []
→ At execution, this resolves to '' (no context injected).


In [10]:
# ── TASK 4 (15 pts): Hierarchical process comparison ────────────────────
# Run the FIX cell above first (patches crewai.llms.cache.mark_cache_breakpoint).

llm = LLM(model="groq/openai/gpt-oss-20b", temperature=0)  # no cache param

urgency_analyst = Agent(
    role="Urgency Analyst",
    goal="Classify urgency as Low, Medium, or High.",
    backstory="You detect urgency from language.",
    llm=llm
)
sentiment_analyst = Agent(
    role="Sentiment Analyst",
    goal="Classify sentiment as Calm, Frustrated, or Angry.",
    backstory="You read emotional tone.",
    llm=llm
)
category_specialist = Agent(
    role="Category Specialist",
    goal="Classify category as Billing, Technical, Account, or Security.",
    backstory="You map issues to categories.",
    llm=llm
)

urgency_task = Task(description="Urgency of: {ticket_text}", expected_output="Low/Medium/High", agent=urgency_analyst)
sentiment_task = Task(description="Sentiment of: {ticket_text}", expected_output="Calm/Frustrated/Angry", agent=sentiment_analyst)
category_task = Task(description="Category of: {ticket_text}", expected_output="Billing/Technical/Account/Security", agent=category_specialist)

crew_hierarchical = Crew(
    agents=[urgency_analyst, sentiment_analyst, category_specialist],
    tasks=[urgency_task, sentiment_task, category_task],
    process=Process.hierarchical,
    manager_llm=llm
)

async def run_hierarchical():
    ticket = TICKETS[7]  # TS-206-07
    print(f"\n--- Hierarchical run on {ticket['ticket_id']} ---")
    print(f"Text: {ticket['ticket_text']}")
    try:
        result = await crew_hierarchical.kickoff_async(inputs={"ticket_text": ticket['ticket_text']})
        executed_tasks = result.tasks_output
        print(f"Number of tasks executed: {len(executed_tasks)}")
        print("Tasks executed:")
        for task_output in executed_tasks:
            print(f"  - {task_output.agent}: {task_output.raw}")
        print(f"Defined tasks: 3, Executed: {len(executed_tasks)}")
    except Exception as e:
        print(f"Error: {e}")

await run_hierarchical()


--- Hierarchical run on TS-206-07 ---
Text: My internet has been down for 51 hours and I have an urgent client call in 20 minutes. This is unacceptable for a Premium customer!


Number of tasks executed: 3
Tasks executed:
  - Crew Manager: High
  - Crew Manager: Angry
  - Crew Manager: Technical
Defined tasks: 3, Executed: 3


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Number: Sequential run on 3 tickets → 9 LLM calls.
        Hierarchical run on TS-206-07 → 3 tasks executed (all three, same as defined).
Reason: Both sequential and hierarchical executed all tasks in this instance. The difference is that sequential follows a fixed order, while hierarchical has a manager that can dynamically decide which tasks to run – though here it chose to run all of them.
Action: For TeleServe’s pipeline, if the manager’s decisions can be tuned to skip irrelevant tasks (e.g., skip sentiment if urgency is already high and the ticket is clearly technical), hierarchical could reduce cost. If no such dynamic decision is needed, sequential is simpler and sufficient. For true concurrency, LangGraph remains the better choice.

---
## Section 4: Scoring Rubric (100 points)

| Task | Points | Criteria |
|---|---|---|
| Task 1 — Env verification | 15 | Exact error type + message reproduced; explains *why* (native provider list) in own words |
| Task 2 — Sequential crew | 30 | 3 agents correctly role/goal/backstory-defined; 3 tickets × 3 outputs printed; `{ticket_text}` interpolation used correctly (not hardcoded per ticket) |
| Task 3 — Context gotcha | 20 | Both context states reproduced exactly; correct explanation of *why* unset leaks context (NOT_SPECIFIED sentinel, not None) |
| Task 4 — Hierarchical | 15 | Manager LLM correctly wired; task/delegation count accurately reported (not assumed) |
| Task 5 — NRA + interview | 20 | Number is a real figure from today's run; Reason is mechanism-language (process type / context chaining), not benefit-language; Action is specific with a named threshold |

**Technical vs. Communication split:** Tasks 1–4 are technical (correct code, correct verification).
Task 5 is communication (NRA discipline) — graded and reported separately per the Month 12/13
standing rule.
